# Grid Saver - MVP Pipeline
# Sense → Predict → Act: Unified Production Architecture
# Author: Justine Adzormado
# Stack: Google Colab + GitHub (Private) + Streamlit Community Cloud

---
> **Purpose:** This notebook builds the Grid Saver MVP pipeline.
> It extracts the validated logic from Phase 1 (Sense), Phase 2 (Predict), and Phase 3 (Act)
> and packages them into clean, deployable functions.
> The trained XGBoost model is saved as `gridsaver_model.pkl`.
> The processed dataset is saved as `data_sample.csv`.
> Both are loaded directly by `app.py`
---

## SECTION 0 - IMPORTS

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score
)

# Mount Google Drive (raw academic data)
from google.colab import drive
drive.mount('/content/drive')

print('Grid Saver - MVP Pipeline')
print('Author: Justine Adzormado')
print('=' * 60)
print('Libraries loaded. Google Drive mounted.')

Mounted at /content/drive
Grid Saver - MVP Pipeline
Author: Justine Adzormado
Libraries loaded. Google Drive mounted.



## SECTION 1 - LOAD RAW DATA
> Raw academic datasets loaded from Google Drive.

In [2]:
# ============================================================
# PATHS — update if your Drive folder structure differs
# ============================================================
ERCOT_PATH = '/content/drive/MyDrive/snapshots_2026-02-10_US-TEX-ERCO-2025-hourly.csv'
PJM_PATH   = '/content/drive/MyDrive/PJM_Load_hourly.csv'
PECAN_PATH = '/content/drive/MyDrive/pecan_sample.csv'

# Column references — same as all three prototype notebooks
CARBON_COL = 'Carbon intensity gCO₂eq/kWh (direct)'
CFE_COL    = 'Carbon-free energy percentage (CFE%)'

# --- ERCOT (Sense Layer) ---
df_ercot = pd.read_csv(ERCOT_PATH)
df_ercot['Datetime (UTC)'] = pd.to_datetime(df_ercot['Datetime (UTC)'])
df_ercot = df_ercot.sort_values('Datetime (UTC)').reset_index(drop=True)

# --- PJM (Predict Layer) ---
df_pjm = pd.read_csv(PJM_PATH)
df_pjm.columns = ['datetime', 'demand_mw']
df_pjm['datetime'] = pd.to_datetime(df_pjm['datetime'])
df_pjm = df_pjm.sort_values('datetime').reset_index(drop=True)

# --- Pecan Street (Act Layer) ---
df_pecan = pd.read_csv(PECAN_PATH)
df_pecan['local_15min'] = pd.to_datetime(df_pecan['local_15min'])
df_pecan = df_pecan.sort_values('local_15min').reset_index(drop=True)

print(f'ERCOT loaded:       {len(df_ercot):,} hourly records')
print(f'PJM loaded:         {len(df_pjm):,} hourly records')
print(f'Pecan Street loaded: {len(df_pecan):,} records')

ERCOT loaded:       8,760 hourly records
PJM loaded:         32,896 hourly records
Pecan Street loaded: 868,096 records


## SECTION 2 - SENSE LAYER
> Validated logic from Phase 1.

In [3]:
# ============================================================
# SENSE LAYER FUNCTION
# ============================================================
# Grid Vulnerability Score Formula (same as Phase 1):
# Score = (carbon_intensity_normalised * 70) + (fossil_share * 30)
# Range: 0 (cleanest) to 100 (most stressed)
# Vulnerability threshold: top 15% most stressed hours

def sense_layer(df):
    """
    Input:  ERCOT dataframe with carbon intensity and CFE% columns
    Output: dataframe with vulnerability_score, vulnerability_event,
            grid_status, hour, month, date, month_name columns added
    """
    df = df.copy()

    df['hour']       = df['Datetime (UTC)'].dt.hour
    df['month']      = df['Datetime (UTC)'].dt.month
    df['date']       = df['Datetime (UTC)'].dt.date
    df['month_name'] = df['Datetime (UTC)'].dt.strftime('%b')

    carbon_max = df[CARBON_COL].max()
    carbon_min = df[CARBON_COL].min()
    cfe_max    = df[CFE_COL].max()

    carbon_denom = (carbon_max - carbon_min) if (carbon_max - carbon_min) != 0 else 1
    cfe_denom    = cfe_max if cfe_max != 0 else 1

    df['vulnerability_score'] = (
        ((df[CARBON_COL] - carbon_min) / carbon_denom * 70) +
        ((1 - df[CFE_COL] / cfe_denom) * 30)
    ).round(1)

    VULNERABILITY_THRESHOLD = df['vulnerability_score'].quantile(0.85)
    df['vulnerability_event'] = df['vulnerability_score'] >= VULNERABILITY_THRESHOLD

    def classify_status(score):
        if score < 40:
            return 'STABLE'
        elif score < 70:
            return 'WARNING'
        else:
            return 'CRITICAL'

    df['grid_status'] = df['vulnerability_score'].apply(classify_status)

    return df, VULNERABILITY_THRESHOLD


# Run Sense Layer
df_ercot, VULNERABILITY_THRESHOLD = sense_layer(df_ercot)

print('Sense Layer complete')
print(f'Vulnerability Threshold: {VULNERABILITY_THRESHOLD:.1f} / 100')
print(f'Vulnerable hours:        {df_ercot["vulnerability_event"].sum():,} '
      f'({df_ercot["vulnerability_event"].mean()*100:.1f}% of year)')
print(f'Grid Status Distribution:')
print(df_ercot['grid_status'].value_counts())

Sense Layer complete
Vulnerability Threshold: 74.6 / 100
Vulnerable hours:        1,316 (15.0% of year)
Grid Status Distribution:
grid_status
WARNING     4341
STABLE      2566
CRITICAL    1853
Name: count, dtype: int64


## SECTION 3 - PREDICT LAYER
> Validated logic from Phase 2. Same feature engineering, same model config,
> same decision threshold. Model saved as gridsaver_model.pkl for app.py.

In [4]:
# ============================================================
# FEATURE ENGINEERING FUNCTION (same as Phase 2)
# ============================================================
def engineer_features(df):
    df = df.copy()
    df['hour']        = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month']       = df['datetime'].dt.month
    df['day_of_year'] = df['datetime'].dt.dayofyear
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    df['is_summer']   = df['month'].isin([6, 7, 8]).astype(int)
    df['is_winter']   = df['month'].isin([12, 1, 2]).astype(int)

    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)

    df['demand_lag_1h']          = df['demand_mw'].shift(1)
    df['demand_lag_2h']          = df['demand_mw'].shift(2)
    df['demand_lag_24h']         = df['demand_mw'].shift(24)
    df['demand_lag_48h']         = df['demand_mw'].shift(48)
    df['demand_lag_168h']        = df['demand_mw'].shift(168)
    df['demand_rolling_6h_mean'] = df['demand_mw'].rolling(6).mean()
    df['demand_rolling_24h_mean']= df['demand_mw'].rolling(24).mean()
    df['demand_rolling_24h_max'] = df['demand_mw'].rolling(24).max()
    df['demand_rolling_24h_std'] = df['demand_mw'].rolling(24).std()
    df['demand_delta_1h']        = df['demand_mw'].diff(1)
    df['demand_delta_24h']       = df['demand_mw'].diff(24)

    df = df.dropna().reset_index(drop=True)
    return df


# ============================================================
# TRAIN AND SAVE MODEL
# ============================================================
# Same config as Phase 2
# Decision threshold: 0.4 (safety-first recall)
# Forecasting horizon: 24 hours ahead
# Vulnerability threshold: top 15% of demand

DECISION_THRESHOLD   = 0.4
PJM_VULN_THRESHOLD   = df_pjm['demand_mw'].quantile(0.85)

# Step 1 - Feature engineering, drop NAs from lag/rolling calculations
df_features = engineer_features(df_pjm)
print(f'Records after feature engineering: {len(df_features):,}')

# Step 2 - Define target variable
df_features['vulnerability_event'] = (
    df_features['demand_mw'] >= PJM_VULN_THRESHOLD
).astype(int)

# Step 3 - Shift label 24 hours ahead (true advance warning)
df_features['vulnerability_event'] = df_features['vulnerability_event'].shift(-24)

# Step 4 - Drop NAs introduced by the label shift separately
df_features = df_features.dropna().reset_index(drop=True)
print(f'Records after label shift dropna: {len(df_features):,}')

# Define features and target first
feature_cols = [c for c in df_features.columns
                if c not in ['datetime', 'demand_mw', 'vulnerability_event']]

X = df_features[feature_cols]
y = df_features['vulnerability_event']

# Calculate scale_pos_weight from full dataset BEFORE split
vulnerable_count  = y.sum()
total_count       = len(y)
scale_pos_weight  = (total_count - vulnerable_count) / vulnerable_count
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

# Split
split_idx = int(len(df_features) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print('Training XGBoost model...')
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

# Evaluate
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred       = (y_pred_proba >= DECISION_THRESHOLD).astype(int)

precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_pred_proba)

print('Predict Layer complete')
print(f'Precision:  {precision:.3f}')
print(f'Recall:     {recall:.3f} - catches {recall*100:.1f}% of real events')
print(f'F1 Score:   {f1:.3f}')
print(f'ROC-AUC:    {auc:.3f}')

# SAVE MODEL - this is the file that goes into the private repo
MODEL_SAVE_PATH = '/content/drive/MyDrive/gridsaver_model.pkl'
joblib.dump(model, MODEL_SAVE_PATH)
print(f'\nModel saved to Drive: {MODEL_SAVE_PATH}')

Records after feature engineering: 32,728
Records after label shift dropna: 32,704
scale_pos_weight = 5.63
Training XGBoost model...
Predict Layer complete
Precision:  0.752
Recall:     0.913 - catches 91.3% of real events
F1 Score:   0.825
ROC-AUC:    0.978

Model saved to Drive: /content/drive/MyDrive/gridsaver_model.pkl


## SECTION 4 - ACT LAYER
> Validated logic from Phase 3.

In [5]:
# ============================================================
# ACT LAYER FUNCTION
# ============================================================
# Reduction rate: 4% HVAC cycling (validated in Phase 1 and Phase 3)
# NUM_HOMES: 25 real Austin TX households (Pecan Street 2018)
# Per home reduction: 0.0920 kW (derived from Phase 1 real peak event data)
# Scale: 1,000,000 homes = 92 MW removed

REDUCTION_RATE = 0.04
NUM_HOMES      = 25
KW_PER_HOME    = 0.0920

def act_layer(df_pecan):
    """
    Input:  Pecan Street dataframe
    Output: hourly HVAC reduction profile per home
    """
    df = df_pecan.copy()
    df['local_15min'] = pd.to_datetime(df['local_15min'], utc=True)
    df['hour']  = df['local_15min'].dt.hour
    df['month'] = df['local_15min'].dt.month

    pecan_hourly = df.groupby(['hour', 'month']).agg(
        avg_grid_kw = ('grid', 'mean'),
        avg_hvac_kw = ('total_hvac_kw', 'mean'),
        max_grid_kw = ('grid', 'max')
    ).reset_index()

    pecan_hourly['hvac_reduction_kw'] = (
        pecan_hourly['avg_hvac_kw'] * REDUCTION_RATE
    ).round(4)

    return pecan_hourly


# Run Act Layer
pecan_hourly = act_layer(df_pecan)

# Validated constant
kw_per_home = KW_PER_HOME

print('Act Layer complete')
print(f'HVAC reduction rate:  {REDUCTION_RATE*100:.0f}%')
print(f'Homes:                {NUM_HOMES}')
print(f'Per home reduction:   {kw_per_home:.4f} kW')
print(f'Scale projection:     {kw_per_home * 1_000_000 / 1000:.0f} MW at 1M homes')

Act Layer complete
HVAC reduction rate:  4%
Homes:                25
Per home reduction:   0.0920 kW
Scale projection:     92 MW at 1M homes


## SECTION 5 - SPA INTEGRATION
> Dual-confirmation logic from Phase 3.
> Grid Saver only acts when BOTH Sense AND Predict independently confirm risk.

In [6]:
# ============================================================
# SPA PIPELINE - FULL INTEGRATION
# ============================================================
# Alignment key: hour + month (consistent across all three datasets)
# Same logic as Phase 3

# --- Sense triggers (from ERCOT) ---
spa_pipeline = df_ercot.copy().reset_index(drop=True)
spa_pipeline['sense_triggered'] = spa_pipeline['vulnerability_event']

# --- Predict triggers (hour + month alignment, same as Phase 3) ---
# Compute average predict probability per hour + month from PJM model
df_features['vuln_proba'] = model.predict_proba(X)[:, 1]

predict_by_hour_month = df_features.groupby(
    [df_features['datetime'].dt.hour.rename('hour'),
     df_features['datetime'].dt.month.rename('month')]
)['vuln_proba'].mean().reset_index()
predict_by_hour_month['predict_triggered'] = (
    predict_by_hour_month['vuln_proba'] >= DECISION_THRESHOLD
)

# Merge predict signal into SPA pipeline by hour + month
spa_pipeline = spa_pipeline.merge(
    predict_by_hour_month[['hour', 'month', 'vuln_proba', 'predict_triggered']],
    on=['hour', 'month'],
    how='left'
)

# --- SPA dual-confirmation logic ---
spa_pipeline['spa_action_triggered'] = (
    spa_pipeline['sense_triggered'] & spa_pipeline['predict_triggered']
)

# --- Merge Act Layer reduction ---
spa_pipeline = spa_pipeline.merge(
    pecan_hourly[['hour', 'month', 'hvac_reduction_kw']],
    on=['hour', 'month'],
    how='left'
)
spa_pipeline['actual_reduction_kw'] = np.where(
    spa_pipeline['spa_action_triggered'],
    spa_pipeline['hvac_reduction_kw'],
    0
)

total_hours   = len(spa_pipeline)
sense_hours   = int(spa_pipeline['sense_triggered'].sum())
predict_hours = int(spa_pipeline['predict_triggered'].sum())
action_hours  = int(spa_pipeline['spa_action_triggered'].sum())

print('SPA Integration complete')
print(f'Sense triggers:    {sense_hours:,} hours ({sense_hours/total_hours*100:.1f}% of year)')
print(f'Predict triggers:  {predict_hours:,} hours ({predict_hours/total_hours*100:.1f}%)')
print(f'Actions triggered: {action_hours:,} hours ({action_hours/total_hours*100:.1f}% of year)')

SPA Integration complete
Sense triggers:    1,316 hours (15.0% of year)
Predict triggers:  1,659 hours (18.9%)
Actions triggered: 154 hours (1.8% of year)


## SECTION 6 - EXPORT PROCESSED DATASET
> This saves the processed ERCOT output as data_sample.csv.

> Which file goes into the private repo and is loaded directly by app.py.

In [7]:
# ============================================================
# EXPORT PROCESSED DATASET FOR APP.PY
# ============================================================
# Only exporting the columns app.py needs (no raw academic columns)

export_cols = [
    'Datetime (UTC)',
    CARBON_COL,
    CFE_COL,
    'hour',
    'month',
    'month_name',
    'date',
    'vulnerability_score',
    'vulnerability_event',
    'grid_status'
]

df_export = df_ercot[export_cols].copy()

DATA_SAVE_PATH = '/content/drive/MyDrive/data_sample.csv'
df_export.to_csv(DATA_SAVE_PATH, index=False)

print(f'Processed dataset saved: {DATA_SAVE_PATH}')
print(f'Rows exported: {len(df_export):,}')
print(f'Columns exported: {list(df_export.columns)}')
print('This file contains only derived outputs (not raw academic data).')

Processed dataset saved: /content/drive/MyDrive/data_sample.csv
Rows exported: 8,760
Columns exported: ['Datetime (UTC)', 'Carbon intensity gCO₂eq/kWh (direct)', 'Carbon-free energy percentage (CFE%)', 'hour', 'month', 'month_name', 'date', 'vulnerability_score', 'vulnerability_event', 'grid_status']
This file contains only derived outputs (not raw academic data).


## SECTION 7 - MVP SUMMARY

In [8]:
# ============================================================
# GRID SAVER - MVP PIPELINE COMPLETE
# ============================================================
print()
print('=' * 60)
print('GRID SAVER - MVP PIPELINE COMPLETE')
print('=' * 60)
print()
print('PIPELINE')
print(f'  Sense Layer:   ERCOT US-TEX-ERCO 2025 - {len(df_ercot):,} hourly records')
print(f'  Predict Layer: PJM XGBoost - {len(df_features):,} records, 24hr ahead')
print(f'  Act Layer:     Pecan Street Austin 2018 - {NUM_HOMES} real households')
print()
print('MODEL PERFORMANCE')
print(f'  Precision:  {precision:.3f}')
print(f'  Recall:     {recall:.3f} - catches {recall*100:.1f}% of real events')
print(f'  F1 Score:   {f1:.3f}')
print(f'  ROC-AUC:    {auc:.3f}')
print()
print('SPA RESULTS')
print(f'  Sense triggers:    {sense_hours:,} hours ({sense_hours/total_hours*100:.1f}% of year)')
print(f'  Predict triggers:  {predict_hours:,} hours ({predict_hours/total_hours*100:.1f}%)')
print(f'  Actions triggered: {action_hours:,} hours ({action_hours/total_hours*100:.1f}% of year)')
print()
print('SCALE PROJECTION')
print(f'  Per home reduction:  {kw_per_home:.4f} kW')
print(f'  1,000,000 homes:     {kw_per_home * 1_000_000 / 1000:.0f} MW removed')
print()
print('OUTPUTS SAVED TO DRIVE')
print(f'  gridsaver_model.pkl  - trained XGBoost model')
print(f'  data_sample.csv      - processed ERCOT dataset')


GRID SAVER - MVP PIPELINE COMPLETE

PIPELINE
  Sense Layer:   ERCOT US-TEX-ERCO 2025 - 8,760 hourly records
  Predict Layer: PJM XGBoost - 32,704 records, 24hr ahead
  Act Layer:     Pecan Street Austin 2018 - 25 real households

MODEL PERFORMANCE
  Precision:  0.752
  Recall:     0.913 - catches 91.3% of real events
  F1 Score:   0.825
  ROC-AUC:    0.978

SPA RESULTS
  Sense triggers:    1,316 hours (15.0% of year)
  Predict triggers:  1,659 hours (18.9%)
  Actions triggered: 154 hours (1.8% of year)

SCALE PROJECTION
  Per home reduction:  0.0920 kW
  1,000,000 homes:     92 MW removed

OUTPUTS SAVED TO DRIVE
  gridsaver_model.pkl  - trained XGBoost model
  data_sample.csv      - processed ERCOT dataset
